# Exploratory Analysis of API Traffic Patterns

This notebook explores patterns in RESTful API traffic to identify potential anomalies and security threats.

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plot styling
plt.style.use('ggplot')
sns.set(style="whitegrid")

# Display settings
%matplotlib inline
pd.set_option('display.max_columns', None)

## Data Collection/Loading

In a real-world scenario, we would load actual API logs. For now, we'll create synthetic data to demonstrate the analysis approach.

In [ ]:
# Create synthetic API log data
np.random.seed(42)

# Generate timestamp range (last 24 hours, 1-minute intervals)
timestamps = pd.date_range(end=pd.Timestamp.now(), periods=1440, freq='1min')

# Define API endpoints
endpoints = [
    '/api/users', '/api/users/{id}', '/api/products', 
    '/api/products/{id}', '/api/orders', '/api/auth/login'
]

# Define HTTP methods
methods = ['GET', 'POST', 'PUT', 'DELETE']

# Define status codes
status_codes = [200, 201, 400, 401, 403, 404, 500]

# Generate random data
n_requests = 5000
data = {
    'timestamp': np.random.choice(timestamps, n_requests),
    'endpoint': np.random.choice(endpoints, n_requests),
    'method': np.random.choice(methods, n_requests, p=[0.6, 0.2, 0.15, 0.05]),  # GET is most common
    'status_code': np.random.choice(status_codes, n_requests, p=[0.75, 0.1, 0.05, 0.03, 0.02, 0.03, 0.02]),
    'response_time_ms': np.random.gamma(shape=2.0, scale=50.0, size=n_requests),
    'user_id': np.random.choice(range(1, 101), n_requests),  # 100 different users
    'ip_address': np.random.choice([f'192.168.1.{i}' for i in range(1, 51)], n_requests)
}

# Create DataFrame
api_logs = pd.DataFrame(data)

# Sort by timestamp
api_logs = api_logs.sort_values('timestamp').reset_index(drop=True)

# Add some SQL query information for PostgreSQL analysis
query_types = ['SELECT', 'INSERT', 'UPDATE', 'DELETE', 'JOIN']
tables = ['users', 'products', 'orders', 'inventory', 'payments']

api_logs['query_type'] = np.random.choice(query_types + [None], n_requests, p=[0.4, 0.1, 0.1, 0.05, 0.15, 0.2])
api_logs['table_accessed'] = np.random.choice(tables + [None], n_requests, p=[0.2, 0.2, 0.2, 0.1, 0.1, 0.2])
api_logs['query_time_ms'] = np.where(api_logs['query_type'].notnull(), 
                                    np.random.gamma(shape=1.5, scale=30.0, size=n_requests), 
                                    np.nan)

# Display first few rows
api_logs.head()

## Add Anomalies

Let's inject some anomalous patterns that might indicate security threats:

In [ ]:
# 1. SQL Injection attempt (suspicious query patterns)
sql_injection_indices = np.random.choice(range(len(api_logs)), size=20, replace=False)
api_logs.loc[sql_injection_indices, 'endpoint'] = '/api/products'
api_logs.loc[sql_injection_indices, 'method'] = 'GET'
api_logs.loc[sql_injection_indices, 'query_type'] = 'SELECT'
api_logs.loc[sql_injection_indices, 'query_time_ms'] = np.random.gamma(shape=5.0, scale=100.0, size=len(sql_injection_indices))

# 2. Brute force login attempts (many failed logins from same IP)
brute_force_indices = range(3000, 3050)  # 50 consecutive requests
api_logs.loc[brute_force_indices, 'endpoint'] = '/api/auth/login'
api_logs.loc[brute_force_indices, 'method'] = 'POST'
api_logs.loc[brute_force_indices, 'status_code'] = 401
api_logs.loc[brute_force_indices, 'ip_address'] = '192.168.1.99'
api_logs.loc[brute_force_indices, 'response_time_ms'] = np.random.uniform(10, 20, size=len(brute_force_indices))

# 3. Unusual access pattern (rapid queries to endpoints)
unusual_indices = range(4000, 4030)  # 30 consecutive requests
api_logs.loc[unusual_indices, 'endpoint'] = '/api/users/{id}'
api_logs.loc[unusual_indices, 'method'] = 'GET'
api_logs.loc[unusual_indices, 'status_code'] = 200
api_logs.loc[unusual_indices, 'ip_address'] = '192.168.1.50'
api_logs.loc[unusual_indices, 'user_id'] = 5  # Single user making many requests

# Save to CSV
api_logs.to_csv('../../data/synthetic/api_logs_synthetic.csv', index=False)

print(f"Generated {len(api_logs)} API log entries with injected anomalies")

## Basic Exploratory Analysis

In [ ]:
# Summary statistics
api_logs.describe()

In [ ]:
# Distribution of HTTP methods
plt.figure(figsize=(10, 6))
sns.countplot(data=api_logs, x='method', palette='viridis')
plt.title('Distribution of HTTP Methods')
plt.xlabel('HTTP Method')
plt.ylabel('Count')
plt.show()

In [ ]:
# Distribution of status codes
plt.figure(figsize=(12, 6))
ax = sns.countplot(data=api_logs, x='status_code', palette='viridis')
plt.title('Distribution of HTTP Status Codes')
plt.xlabel('Status Code')
plt.ylabel('Count')

# Add percentage labels
total = len(api_logs)
for p in ax.patches:
    percentage = f'{100 * p.get_height() / total:.1f}%'
    ax.annotate(percentage, (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom')

plt.show()

In [ ]:
# Requests over time
api_logs['hour'] = api_logs['timestamp'].dt.hour

plt.figure(figsize=(14, 6))
hourly_counts = api_logs.groupby('hour').size()
sns.lineplot(x=hourly_counts.index, y=hourly_counts.values)
plt.title('API Requests by Hour')
plt.xlabel('Hour of Day')
plt.ylabel('Number of Requests')
plt.xticks(range(24))
plt.grid(True)
plt.show()

In [ ]:
# Response time distribution
plt.figure(figsize=(12, 6))
sns.histplot(api_logs['response_time_ms'], bins=50, kde=True)
plt.title('Distribution of API Response Times')
plt.xlabel('Response Time (ms)')
plt.ylabel('Frequency')
plt.show()

## Anomaly Detection Features

Let's create some features that might be useful for detecting anomalies:

In [ ]:
# 1. Request frequency by IP address
ip_request_counts = api_logs.groupby('ip_address').size().reset_index(name='request_count')
plt.figure(figsize=(14, 6))
sns.histplot(ip_request_counts['request_count'], bins=30)
plt.title('Distribution of Request Counts by IP Address')
plt.xlabel('Number of Requests')
plt.ylabel('Number of IP Addresses')
plt.show()

# Find outliers (IPs with high request counts)
high_frequency_ips = ip_request_counts.sort_values('request_count', ascending=False).head(5)
print("Top 5 IPs by request frequency:")
print(high_frequency_ips)

In [ ]:
# 2. Failed authentication attempts by IP
auth_failures = api_logs[
    (api_logs['endpoint'] == '/api/auth/login') & 
    (api_logs['status_code'] == 401)
].groupby('ip_address').size().reset_index(name='failed_attempts')

plt.figure(figsize=(12, 6))
auth_failures_sorted = auth_failures.sort_values('failed_attempts', ascending=False)
sns.barplot(data=auth_failures_sorted.head(10), x='ip_address', y='failed_attempts')
plt.title('Top 10 IP Addresses by Failed Authentication Attempts')
plt.xlabel('IP Address')
plt.ylabel('Number of Failed Attempts')
plt.xticks(rotation=45)
plt.show()

In [ ]:
# 3. Query time analysis for potential SQL injection
query_times = api_logs[api_logs['query_time_ms'].notnull()].copy()

plt.figure(figsize=(12, 6))
sns.boxplot(data=query_times, x='query_type', y='query_time_ms')
plt.title('Query Times by Query Type')
plt.xlabel('Query Type')
plt.ylabel('Query Time (ms)')
plt.yscale('log')  # Log scale helps visualize outliers
plt.show()

# Find unusually long-running queries
outlier_threshold = query_times['query_time_ms'].quantile(0.99)
outlier_queries = query_times[query_times['query_time_ms'] > outlier_threshold]
print(f"Queries with execution time > {outlier_threshold:.2f}ms (99th percentile):")
print(outlier_queries[['timestamp', 'endpoint', 'method', 'query_type', 'table_accessed', 'query_time_ms']].head())

In [ ]:
# 4. Time-based analysis - request velocity
# Calculate time difference between consecutive requests for each IP
api_logs_sorted = api_logs.sort_values(['ip_address', 'timestamp'])
api_logs_sorted['prev_timestamp'] = api_logs_sorted.groupby('ip_address')['timestamp'].shift(1)
api_logs_sorted['time_delta'] = (api_logs_sorted['timestamp'] - api_logs_sorted['prev_timestamp']).dt.total_seconds()

# Remove first request for each IP (no previous timestamp)
api_logs_sorted = api_logs_sorted.dropna(subset=['time_delta'])

# Calculate average time between requests for each IP
ip_velocity = api_logs_sorted.groupby('ip_address')['time_delta'].agg(['mean', 'min', 'count']).reset_index()
ip_velocity = ip_velocity[ip_velocity['count'] > 5]  # Only IPs with more than 5 requests

# Display IPs with suspiciously rapid requests
suspicious_velocity = ip_velocity.sort_values('min').head(10)
print("IPs with potentially suspicious request patterns (small time between requests):")
print(suspicious_velocity)

## Next Steps

Based on this exploratory analysis, we can identify several potentially useful features for anomaly detection:

1. Request frequency by IP address
2. Failed authentication attempts
3. Query execution times
4. Time between consecutive requests (request velocity)
5. Unusual endpoint access patterns
6. Status code distribution per IP/user

In the next notebook, we'll engineer these features and apply anomaly detection algorithms to identify potential security threats.